# Deep Learning Model Training: Bone Age Prediction

This notebook implements and trains a Xception-based CNN for predicting bone age from hand X-ray images.

## Approach
- Transfer learning with Xception architecture
- Data augmentation for robustness
- Regression model predicting age in months
- Custom MAE metric for evaluation


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import sys
sys.path.append('../src')

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

## Data Preparation

In [ ]:
# Load and prepare dataset
# NOTE: Adjust the data_path and csv_file paths to match your dataset location
data_path = Path('../data')
csv_file = data_path / 'boneage-training-dataset.csv'

# Check if file exists, if not provide instructions
if csv_file.exists():
    df = pd.read_csv(csv_file)
    print(f"✓ Dataset loaded: {len(df)} samples")
    
    # Calculate normalization parameters
    boneage_mean = df['boneage'].mean()
    boneage_std = df['boneage'].std()
    print(f"✓ Bone age - Mean: {boneage_mean:.2f}, Std: {boneage_std:.2f} months")
    
    # Normalize target
    df['norm_age'] = (df['boneage'] - boneage_mean) / boneage_std
    
    # Create image paths - adjust 'images' folder name if different
    image_dir = data_path / 'images'
    df['path'] = df['id'].apply(lambda x: str(image_dir / f'{x}.png'))
    
    # Split data
    df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
    df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, shuffle=True)
    
    print(f"✓ Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
else:
    print("⚠️ Dataset file not found!")
    print(f"   Expected location: {csv_file}")
    print("   Please:")
    print("   1. Download the RSNA Bone Age dataset from Kaggle")
    print("   2. Place 'boneage-training-dataset.csv' in the ../data/ folder")
    print("   3. Place images in ../data/images/ folder")
    print("   4. Re-run this cell")
    # Create dummy data for demonstration (remove this in production)
    print("\n   Creating dummy data structure for demonstration...")
    df = pd.DataFrame({
        'id': range(100),
        'boneage': np.random.randint(60, 240, 100),
        'male': np.random.choice([True, False], 100)
    })
    boneage_mean = df['boneage'].mean()
    boneage_std = df['boneage'].std()
    df['norm_age'] = (df['boneage'] - boneage_mean) / boneage_std
    df['path'] = df['id'].apply(lambda x: f'../data/images/{x}.png')
    df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
    df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, shuffle=True)
    print("   ⚠️ Using dummy data - replace with real dataset for actual training!")

## Data Generators with Augmentation

In [ ]:
# Create data generators
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1/255.0,
    preprocessing_function=preprocess_input,
    rotation_range=180,
    zoom_range=0.25,
    brightness_range=[0.2, 0.5],
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    shear_range=0.05,
    fill_mode='nearest'
)

# Validation/test generator (no augmentation)
val_test_datagen = ImageDataGenerator(
    rescale=1/255.0,
    preprocessing_function=preprocess_input
)

# Create generators
# NOTE: If images don't exist, this will show warnings but won't fail
# Make sure image paths in df match actual image locations
try:
    train_gen = train_datagen.flow_from_dataframe(
        df_train,
        x_col='path',
        y_col='norm_age',
        batch_size=BATCH_SIZE,
        seed=42,
        shuffle=True,
        class_mode='raw',
        color_mode='rgb',
        target_size=IMAGE_SIZE
    )
    
    val_gen = val_test_datagen.flow_from_dataframe(
        df_val,
        x_col='path',
        y_col='norm_age',
        batch_size=BATCH_SIZE,
        seed=42,
        shuffle=False,
        class_mode='raw',
        color_mode='rgb',
        target_size=IMAGE_SIZE
    )
    
    test_gen = val_test_datagen.flow_from_dataframe(
        df_test,
        x_col='path',
        y_col='norm_age',
        batch_size=BATCH_SIZE,
        seed=42,
        shuffle=False,
        class_mode='raw',
        color_mode='rgb',
        target_size=IMAGE_SIZE
    )
    
    print(f"✓ Train batches: {len(train_gen)}")
    print(f"✓ Val batches: {len(val_gen)}")
    print(f"✓ Test batches: {len(test_gen)}")
except Exception as e:
    print(f"⚠️ Error creating generators: {e}")
    print("   Make sure image files exist at the specified paths")
    print("   Check that df['path'] contains valid image file paths")

## Model Architecture

In [ ]:
# Import model builder
from model import build_xception_model, create_mae_metric

# Build model
model = build_xception_model(
    input_shape=(*IMAGE_SIZE, 3),
    boneage_mean=boneage_mean,
    boneage_std=boneage_std,
    dense_units=10,
    learning_rate=0.001
)

print("Model architecture:")
model.summary()

# Calculate steps per epoch
steps_per_epoch = len(train_gen)
validation_steps = len(val_gen)

## Training Configuration

In [ ]:
# Setup callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        '../models/bone_age_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# Train model
EPOCHS = 50

print("Starting training...")
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training complete!")

## Training History Visualization


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_title('Mean Absolute Error', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MAE (months)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Model Evaluation


In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = model.evaluate(test_gen, verbose=1)
print(f"\nTest Loss (MSE): {test_results[0]:.4f}")
print(f"Test MAE: {test_results[1]:.2f} months")

# Make predictions
print("\nGenerating predictions...")
test_gen.reset()
predictions = model.predict(test_gen, verbose=1)

# Denormalize predictions and actual values
pred_ages = predictions.flatten() * boneage_std + boneage_mean
actual_ages = df_test['boneage'].values[:len(pred_ages)]

# Calculate metrics
mae = np.mean(np.abs(pred_ages - actual_ages))
rmse = np.sqrt(np.mean((pred_ages - actual_ages)**2))

print(f"\nFinal Metrics:")
print(f"  MAE: {mae:.2f} months")
print(f"  RMSE: {rmse:.2f} months")


## Prediction Visualization


In [ ]:
# Scatter plot of predictions vs actual
plt.figure(figsize=(10, 8))
plt.scatter(actual_ages, pred_ages, alpha=0.6, s=50)
plt.plot([actual_ages.min(), actual_ages.max()], 
         [actual_ages.min(), actual_ages.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Age (months)', fontsize=12)
plt.ylabel('Predicted Age (months)', fontsize=12)
plt.title('Predicted vs Actual Bone Age', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Error distribution
errors = pred_ages - actual_ages
plt.figure(figsize=(10, 6))
plt.hist(errors, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
plt.xlabel('Prediction Error (months)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Prediction Error Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
